# The Double Hadamard Test

An entangled unitary operator can be decomposed using a space-like cut of the form:

$$\Gamma := {(c_i,  V_i \oplus W_i) : i \hspace{1mm} \epsilon \hspace{1mm} [m]}$$

For some positive integer, m & random variables, (i, j, g) such that:  

$$ {i, j \hspace{1mm} \epsilon \hspace{1mm} m \hspace{1mm} | \hspace{1mm} c_i, \hspace{1mm} m > 0}$$

and 

$${g \hspace{1mm} \epsilon \hspace{1mm} (0, 1)} $$

In [6]:
import cudaq
import numpy as np
import random
import pennylane as qml

# Set up the CUDA-Q target for multiple QPUs if available
cudaq.set_target("nvidia", option="mqpu")
print(f"Number of QPUs available: {cudaq.get_target().num_qpus()}")

Number of QPUs available: 1


We substitute a controlled two-qubit pauli from each ancilla to both qubits, in-place of the unitary.

$$ V_i / V_j = \ket{0}\bra{0} \oplus V_i + \ket{1}\bra{1} \oplus V_j$$

In [7]:
pauli_names = ['I', 'X', 'Y', 'Z']

def parse_pauli_string(pauli_string):
    """Extracts valid Pauli characters from a string (e.g., 'Identity' -> 'I')."""
    parsed = ""
    for char in pauli_string:
        if char in pauli_names:
            parsed += char
    return parsed

def apply_pauli_gate(kernel, theta, pauli, target_qubit, control_qubit):
    """Apply the specified Pauli gate to the given qubit index."""
    if pauli == 'I':  # I
        pass  # Identity, do nothing
    elif pauli == 'X':  # X
        kernel.crx(theta, target_qubit, control_qubit)
        #kernel.rx(theta, target_qubit)
    elif pauli == 'Y':  # Y
        kernel.cry(theta, target_qubit, control_qubit)
    elif pauli == 'Z':  # Z
        kernel.crz(theta, target_qubit, control_qubit)
        #kernel.rz(theta, target_qubit)
    return kernel

def LCU_terms(LCU_ops):
    # Initialize with an empty string to allow concatenation
    result1 = [""]
    result2 = [""]
    
    for op1, op2 in LCU_ops:
        char1 = parse_pauli_string(str(op1))
        char2 = parse_pauli_string(str(op2))
        
        # For every existing combination, create a new one with the current Pauli appended
        # result1 += [existing_string + char1]
        result1 += [curr + char1 for curr in result1]
        result2 += [curr + char2 for curr in result2]
    
    # Optional: If you want to remove the initial empty string [""]
    # result1 = result1[1:]
    # result2 = result2[1:]

    print(f"Subcircuit 1 combinations: {result1}")
    print(f"Subcircuit 2 combinations: {result2}")
    
    return [result1, result2]
           
def double_hadamard_test(circuit, index_to_cut, paulis, g):
    num_qubits = len(circuit)
    sublength = num_qubits - index_to_cut
    theta = .5
    shots=1000
    
    # Register custom unitary operations (all gates except the one being cut)
    base_name = 'Cx'
    for i in range(len(circuit)):
        if i != index_to_cut:  # Skip the gate at the cut index
            cudaq.register_operation(f'{base_name}_{i}', circuit[i])

    # Build 1st subcircuit
    # Defining kernel using the builder function
    kernel1 = cudaq.make_kernel() 
    q1 = kernel1.qalloc(index_to_cut + 2) # +1 for zero-index, +1 for ancilla
       
    kernel1.h(q1[0])
    if g != 1:
        kernel1.s(q1[0])

    # Apply the custom operations to the first subcircuit
    for i in range(1, index_to_cut+1):
        kernel1.__getattr__(f'{base_name}_{i-1}')(q1[i], q1[i+1])
       
    # Build 2nd subcircuit 
    kernel2 = cudaq.make_kernel()
    q2 = kernel2.qalloc(sublength + 1)
        
    kernel2.h(q2[sublength])
    if g != 1:
        kernel2.s(q2[sublength])
    
    for gate in paulis[0]:
    # Apply all decomposition gates to the circuit
        kernel1 = apply_pauli_gate(kernel1, theta, gate, q1[0], q1[index_to_cut+1])

    for gate in paulis[1]:
        kernel2 = apply_pauli_gate(kernel2, theta, gate, q2[sublength], q2[0])

     # Apply the remaining custom operations to the second subcircuit
    for i in range(index_to_cut+1, len(circuit)):
        j = i - (index_to_cut + 1) # offset the index to start at 0
        kernel2.__getattr__(f'{base_name}_{i}')(q2[j], q2[j+1])
        
    kernel1.h(q1[0])
    kernel1.mz(q1)

    kernel2.h(q2[sublength])
    kernel2.mz(q2)

    results1 = cudaq.sample(kernel1, shots_count=shots)
    results2 = cudaq.sample(kernel2, shots_count=shots)
    
    if ((results1.expectation() + results2.expectation()) / 2 >= .85):
        print(cudaq.draw(kernel1))
        print(cudaq.draw(kernel2))
    
    
    return [results1, results2]

# Double Hadamard Test File

Observe a base circuit and pass it to the double hadamard test
 - Compute expectation value of circuit
 - Consider Ising models and other observable custom unitaries

 M = (.12) XY + (.34) YZ

In [8]:
cnot_gate = np.array([[1, 0, 0, 0],[0, 1, 0, 0],[0, 0, 0, 1],[0, 0, 1, 0]])
circuit = [cnot_gate, cnot_gate, cnot_gate, cnot_gate, cnot_gate]  # 5 CNOT gates
index_to_cut = 2 # qubit_2 (3rd index)
shots = 1000

base_name = 'Cx'
for i in range(len(circuit)):
    cudaq.register_operation(f'{base_name}_{i}', circuit[i])

# Construct the base kernel
kernel = cudaq.make_kernel()
q = kernel.qalloc(len(circuit) + 1)

for i in range(len(circuit)):
    kernel.__getattr__(f'{base_name}_{i}')(q[i], q[i+1])
    
base_results = cudaq.sample(kernel, shots_count = 1000)

# Decompose the target unitary into a summation of single pauli gates
LCU = qml.pauli_decompose(circuit[index_to_cut])
LCU_coeffs, LCU_ops = LCU.terms()

print(f"LCU decomposition:\n {LCU}")
print(f"Coefficients:\n {LCU_coeffs}")
print(f"Unitaries:\n {LCU_ops}")
subcircuits = LCU_terms(LCU_ops)

for pauli_string1 in subcircuits[0]:
    for pauli_string2 in subcircuits[1]:
        for g in [0, 1]:
            if (len(pauli_string1) > 0 or len(pauli_string2) > 0): # skip empty inputs
                print("\n\n-------------------------\n\n")
                print(f"Results for subcircuits with ({pauli_string1}, {pauli_string2}) & g = {g}")
                results = double_hadamard_test(circuit, index_to_cut, [pauli_string1, pauli_string2], g)
                
                avg = (results[0].expectation() + results[1].expectation()) / 2
                print(f"Expectation values:\n Original Circuit:   {base_results.expectation():.4f}")

                print(f" Subcircuit 1:       {results[0].expectation():.4f}")
                print(f" Subcircuit 2:       {results[1].expectation():.4f}")
                print(f" Subcircuit average: {avg:.4f}")
            

LCU decomposition:
 0.5 * (I(0) @ I(1)) + 0.5 * (I(0) @ X(1)) + 0.5 * (Z(0) @ I(1)) + -0.5 * (Z(0) @ X(1))
Coefficients:
 [ 0.5  0.5  0.5 -0.5]
Unitaries:
 [I(0) @ I(1), I(0) @ X(1), Z(0) @ I(1), Z(0) @ X(1)]
Subcircuit 1 combinations: ['', 'I', 'I', 'II', 'Z', 'IZ', 'IZ', 'IIZ', 'Z', 'IZ', 'IZ', 'IIZ', 'ZZ', 'IZZ', 'IZZ', 'IIZZ']
Subcircuit 2 combinations: ['', 'I', 'X', 'IX', 'I', 'II', 'XI', 'IXI', 'X', 'IX', 'XX', 'IXX', 'IX', 'IIX', 'XIX', 'IXIX']


-------------------------


Results for subcircuits with (, I) & g = 0
Expectation values:
 Original Circuit:   1.0000
 Subcircuit 1:       -0.0420
 Subcircuit 2:       0.0240
 Subcircuit average: -0.0090


-------------------------


Results for subcircuits with (, I) & g = 1
      ╭───╮   ╭───╮  
q0 : ─┤ h ├───┤ h ├──
     ╭┴───┴─╮ ╰───╯  
q1 : ┤      ├────────
     │ Cx_0 │╭──────╮
q2 : ┤      ├┤      ├
     ╰──────╯│ Cx_1 │
q3 : ────────┤      ├
             ╰──────╯

     ╭──────╮        
q0 : ┤      ├────────
     │ Cx_3 │╭──────